# 01 — Data Exploration

Objective: understand the Titanic train/test data before any modelling. Every plot answers a question; measured numbers are quoted so this reads correctly on GitHub without re-running.

Headline facts (measured): train **891** rows, test **418** rows, survival rate **38.38%** (342 survived / 549 died), majority baseline **0.6162**.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import set_global_seed
from configs.config import RANDOM_STATE
set_global_seed(RANDOM_STATE)
%matplotlib inline

## Load and validate schemas


In [ ]:
from src.data_loading import load_raw, missing_value_report, dataset_summary
train_df, test_df = load_raw(ROOT / 'data' / 'raw' / 'train.csv', ROOT / 'data' / 'raw' / 'test.csv')
print(dataset_summary(train_df, test_df))
print(train_df.shape, test_df.shape)
print(train_df.dtypes)

## Missing values, duplicates, types
- Cabin 77.10% train / 78.23% test → derive deck + known-flag, never impute raw.
- Age 19.87% / 20.57% → median imputation in-pipeline.
- Embarked 2 missing (0.22%), both 1st-class survivors (fare 80, cabin B28).
- Fare 1 missing in test (PassengerId 1044, 3rd-class Mr. Storey).
- 0 exact duplicates. `PassengerId` is an identifier, never a feature.


In [ ]:
print(missing_value_report(train_df))
print(missing_value_report(test_df))
print('train dupes:', train_df.duplicated().sum(), '| test dupes:', test_df.duplicated().sum())
print(train_df[train_df['Embarked'].isna()])
print(test_df[test_df['Fare'].isna()][['PassengerId','Pclass','Name','Sex','Age']])

## Target distribution
Imbalanced but not extreme (38% positive): accuracy is a fair competition metric, and the 0.6162 majority baseline is the bar every model must clear.


In [ ]:
from src.visualization import plot_target_distribution
plot_target_distribution(train_df, ROOT / 'reports' / 'figures' / '01_target_distribution.png')
print(train_df['Survived'].value_counts(), train_df['Survived'].mean())

## Survival by Sex / Pclass / Embarked
Measured: female **74.20%** (n=314) vs male **18.89%** (n=577); Pclass 1 **62.96%** / 2 **47.28%** / 3 **24.24%**; Embarked C **55.36%** / Q **38.96%** / S **33.70%**.


In [ ]:
from src.visualization import plot_survival_by_category
for col, name in [('Sex','02_survival_by_sex'), ('Pclass','03_survival_by_pclass'), ('Embarked','08_survival_by_embarked')]:
    print(train_df.groupby(col)['Survived'].agg(['mean','count']))
    plot_survival_by_category(train_df, col, ROOT/'reports'/'figures'/f'{name}.png', f'Survival rate by {col}')

## Age and Fare
Fare median: survived **26.0** vs died **10.5** — strong signal. Age medians are both **28.0**; the KDE shows the real story (child bump among survivors, wide overlap otherwise), so age helps but never alone.


In [ ]:
from src.visualization import plot_age_distribution, plot_fare_distribution
print(train_df.groupby('Survived')[['Age','Fare']].median())
plot_age_distribution(train_df, ROOT/'reports'/'figures'/'04_age_by_survival.png')
plot_fare_distribution(train_df, ROOT/'reports'/'figures'/'05_fare_by_survival.png')

## Family, interactions, correlations
Temporary `FamilySize = SibSp + Parch + 1`: survival is 53.6% at SibSp=1 and 55–60% at Parch=1–3, collapsing for very large families. Sex×Pclass is the sharpest interaction: 1st-class female **96.81%** vs 3rd-class male **13.54%**. Treat correlations as associations, not causes.


In [ ]:
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
print(train_df.groupby('SibSp')['Survived'].mean())
print(train_df.groupby('Parch')['Survived'].mean())
print(train_df.groupby(['Sex','Pclass'])['Survived'].mean())
from src.visualization import plot_correlation_heatmap
plot_correlation_heatmap(train_df, ROOT/'reports'/'figures'/'06_correlation_heatmap.png')
train_df.drop(columns=['FamilySize'], inplace=True)

## Conclusions
1. Sex, Pclass, Fare, Title-family signals dominate; age is secondary. 2. Missingness is structural (Cabin) vs imputable (Age/Fare/Embarked). 3. No duplicates, no schema drift, no label leak into test. Proceed to feature engineering with Title, family, ticket-group, and cabin-derived features.
